<a href="https://colab.research.google.com/github/Zaib919451/22108227/blob/main/Welcome_To_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files
import os
import zipfile
# Step 1: Upload dataset (zip file of images)
print("Upload a ZIP file containing the image dataset (e.g., 'images.zip'):")
uploaded = files.upload()
zip_file = list(uploaded.keys())[0]
# Step 2: Unzip the uploaded dataset
dataset_path = "dataset_images"
os.makedirs(dataset_path, exist_ok=True)
with zipfile.ZipFile(zip_file, 'r') as zip_ref:
zip_ref.extractall(dataset_path)
# Step 3: Upload query image
print("Upload a query image (e.g., 'query.jpg'):")
uploaded_query = files.upload()
query_path = list(uploaded_query.keys())[0]
# Step 4: Histogram extraction function
def get_histogram(img):
hist = cv2.calcHist([img], [0, 1, 2], None, [8, 8, 8], [0, 256, 0, 256, 0, 256])
cv2.normalize(hist, hist)
return hist.flatten()
# Step 5: Read query image and extract histogram
query = cv2.imread(query_path)
query = cv2.resize(query, (200, 200)) # Resize for consistency
query_hist = get_histogram(query)
# Step 6: Compare with dataset images
scores = []
for file in os.listdir(dataset_path):
if file.lower().endswith(('.jpg', '.jpeg', '.png')):
image_path = os.path.join(dataset_path, file)
image = cv2.imread(image_path)
image = cv2.resize(image, (200, 200))
hist = get_histogram(image)
score = cv2.compareHist(query_hist, hist, cv2.HISTCMP_CORREL)
scores.append((score, file, image))
# Step 7: Show top 3 matching images
scores.sort(reverse=True)
top_matches = scores[:3]
print("Top 3 Matching Images:")
plt.figure(figsize=(12, 6))
# Show query image
plt.subplot(1, 4, 1)
plt.imshow(cv2.cvtColor(query, cv2.COLOR_BGR2RGB))
plt.title("Query Image")
plt.axis('off')
# Show top 3 results
for i, (score, fname, img) in enumerate(top_matches):
plt.subplot(1, 4, i + 2)
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.title(f"Match {i+1}\nScore: {score:.2f}")
plt.axis('off')
plt.tight_layout()
plt.show()

IndentationError: expected an indented block after 'with' statement on line 14 (4088020147.py, line 15)

In [2]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
# Function to extract a color histogram from an image in HSV color space
def extract_color_histogram(image, bins=(8, 8, 8)):
hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV) # Convert BGR to HSV
# Compute 3D histogram for H, S, and V channels
hist = cv2.calcHist([hsv], [0, 1, 2], None, bins, [0,180, 0,256, 0,256])
return hist.flatten() # Flatten to 1D array for similarity comparison
# Load query image (image to be matched)
img = cv2.imread('tiger.jpg')
query_features = extract_color_histogram(img) # Extract histogram features of query image
# Load dataset images (candidate matches)
dataset = []
dataset.append(cv2.imread('tp.jpg')) # Example: tiger profile
dataset.append(cv2.imread('fc.jpg')) # Example: forest cat
dataset.append(cv2.imread('lioness.jpg')) # Example: lioness
# Extract histogram features for all dataset images
dataset_features = [extract_color_histogram(image) for image in dataset]
# Compute cosine similarity between query image and each dataset image
similarities = [cosine_similarity([query_features], [features])[0][0] for features in dataset_features]
# Rank images based on similarity score (higher is more similar)
ranked_indices = sorted(range(len(similarities)), key=lambda i: similarities[i], reverse=True)
# Display the query image
plt.figure()
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)) # Convert BGR to RGB for matplotlib
display
plt.title('Query Image')
plt.axis('off')
plt.show()
# Display top-k similar images from the dataset
top_k = 3
for i in range(top_k):
similar_img = dataset[ranked_indices[i]]
plt.figure()
plt.imshow(cv2.cvtColor(similar_img, cv2.COLOR_BGR2RGB))
plt.title(f"Similar Image {i+1}")
plt.axis('off')
plt.show()

IndentationError: expected an indented block after function definition on line 6 (2794297106.py, line 7)

In [3]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage.feature import local_binary_pattern
from google.colab import files
import os
import zipfile
# Step 1: Upload ZIP folder containing dataset
print("Upload a ZIP file of texture images (e.g., 'fabric_textures.zip'):")
uploaded = files.upload()
zip_file = list(uploaded.keys())[0]
dataset_path = "texture_dataset"
os.makedirs(dataset_path, exist_ok=True)
# Step 2: Unzip the dataset
with zipfile.ZipFile(zip_file, 'r') as zip_ref:
zip_ref.extractall(dataset_path)
# Step 3: Upload query image
print("Upload a query texture image (e.g., 'query.jpg'):")
uploaded_query = files.upload()
query_img_path = list(uploaded_query.keys())[0]
# Step 4: LBP feature extraction function
def extract_lbp_features(gray_img):
lbp = local_binary_pattern(gray_img, P=24, R=3, method="uniform")
hist, _ = np.histogram(lbp.ravel(), bins=np.arange(0, 27), range=(0, 26))
hist = hist.astype("float")
hist /= (hist.sum() + 1e-6)
return hist
# Step 5: Extract LBP features from query image
query_img = cv2.imread(query_img_path, cv2.IMREAD_GRAYSCALE)
query_img = cv2.resize(query_img, (256, 256)) # Resize for consistency
query_hist = extract_lbp_features(query_img)
# Step 6: Compare with dataset images using Chi-square distance
def chi2_distance(histA, histB, eps=1e-6):
return 0.5 * np.sum(((histA - histB) ** 2) / (histA + histB + eps))
scores = []
for file in os.listdir(dataset_path):
if file.lower().endswith(('.jpg', '.jpeg', '.png')):
img_path = os.path.join(dataset_path, file)
img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
img = cv2.resize(img, (256, 256))
hist = extract_lbp_features(img)
dist = chi2_distance(query_hist, hist)
scores.append((dist, file, img))
# Step 7: Display top 3 matches
scores.sort(key=lambda x: x[0]) # lower distance = better match
top_matches = scores[:3]
plt.figure(figsize=(12, 6))
plt.subplot(1, 4, 1)
plt.imshow(query_img, cmap='gray')
plt.title("Query Image")
plt.axis('off')
for i, (dist, fname, img) in enumerate(top_matches):
plt.subplot(1, 4, i + 2)
plt.imshow(img, cmap='gray')
plt.title(f"Match {i+1}\nDist: {dist:.2f}")
plt.axis('off')
plt.tight_layout()
plt.show()


IndentationError: expected an indented block after 'with' statement on line 15 (3762039955.py, line 16)

In [4]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
import os
# Upload query image
print("Upload the query image:")
uploaded_query = files.upload()
query_path = list(uploaded_query.keys())[0]
# Upload dataset images
print("Upload dataset images (upload multiple files):")
uploaded_dataset = files.upload()
# Function to extract color histogram (for fashion images)
def extract_color_histogram(image, bins=(8, 8, 8)):
hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
hist = cv2.calcHist([hsv], [0, 1, 2], None, bins, [0,180, 0,256, 0,256])
cv2.normalize(hist, hist)
return hist.flatten()
# Function to extract grayscale histogram (for medical images)
def extract_gray_histogram(image, bins=256):
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
hist = cv2.calcHist([gray], [0], None, [bins], [0, 256])
cv2.normalize(hist, hist)
return hist.flatten()
# Load query image
query_img = cv2.imread(query_path)
# For Fashion E-Commerce, use color histogram:
query_features_fashion = extract_color_histogram(query_img)
# For Medical Imaging, use grayscale histogram:
query_features_medical = extract_gray_histogram(query_img)
# Load dataset images
dataset_imgs = []
dataset_fashion_features = []
dataset_medical_features = []
for filename in uploaded_dataset.keys():
img = cv2.imread(filename)
dataset_imgs.append(img)
dataset_fashion_features.append(extract_color_histogram(img))
dataset_medical_features.append(extract_gray_histogram(img))
# Choose mode: 'fashion' or 'medical'
mode = 'fashion' # change to 'medical' for medical images
if mode == 'fashion':
query_features = query_features_fashion
dataset_features = dataset_fashion_features
elif mode == 'medical':
query_features = query_features_medical
dataset_features = dataset_medical_features
# Calculate cosine similarity scores
similarities = [cosine_similarity([query_features], [feat])[0][0] for feat in dataset_features]
# Sort by similarity descending
ranked_indices = sorted(range(len(similarities)), key=lambda i: similarities[i], reverse=True)
# Display query image
plt.figure(figsize=(5,5))
plt.imshow(cv2.cvtColor(query_img, cv2.COLOR_BGR2RGB))
plt.title('Query Image')
plt.axis('off')
plt.show()
# Display top 3 similar images
top_k = 3
for i in range(min(top_k, len(dataset_imgs))):
idx = ranked_indices[i]
plt.figure(figsize=(5,5))

plt.imshow(cv2.cvtColor(dataset_imgs[idx], cv2.COLOR_BGR2RGB))
plt.title(f"Similar Image {i+1} (Score: {similarities[idx]:.2f})")
plt.axis('off')
plt.show()

IndentationError: expected an indented block after function definition on line 14 (2489452708.py, line 15)

In [6]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
import os
# Upload query image
print("Upload the query image:")
uploaded_query = files.upload()
query_path = list(uploaded_query.keys())[0]
# Upload dataset images
print("Upload dataset images (upload multiple files):")
uploaded_dataset = files.upload()
# Function to extract color histogram (for fashion images)
def extract_color_histogram(image, bins=(8, 8, 8)):
hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
hist = cv2.calcHist([hsv], [0, 1, 2], None, bins, [0,180, 0,256, 0,256])
cv2.normalize(hist, hist)
return hist.flatten()
# Function to extract grayscale histogram (for medical images)
def extract_gray_histogram(image, bins=256):
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
hist = cv2.calcHist([gray], [0], None, [bins], [0, 256])
cv2.normalize(hist, hist)
return hist.flatten()
# Load query image
query_img = cv2.imread(query_path)
# For Fashion E-Commerce, use color histogram:
query_features_fashion = extract_color_histogram(query_img)
# For Medical Imaging, use grayscale histogram:
query_features_medical = extract_gray_histogram(query_img)
# Load dataset images
dataset_imgs = []
dataset_fashion_features = []
dataset_medical_features = []
for filename in uploaded_dataset.keys():
img = cv2.imread(filename)
dataset_imgs.append(img)
dataset_fashion_features.append(extract_color_histogram(img))
dataset_medical_features.append(extract_gray_histogram(img))
# Choose mode: 'fashion' or 'medical'
mode = 'fashion' # change to 'medical' for medical images
if mode == 'fashion':
query_features = query_features_fashion
dataset_features = dataset_fashion_features
elif mode == 'medical':
query_features = query_features_medical
dataset_features = dataset_medical_features
# Calculate cosine similarity scores
similarities = [cosine_similarity([query_features], [feat])[0][0] for feat in dataset_features]
# Sort by similarity descending
ranked_indices = sorted(range(len(similarities)), key=lambda i: similarities[i], reverse=True)
# Display query image
plt.figure(figsize=(5,5))
plt.imshow(cv2.cvtColor(query_img, cv2.COLOR_BGR2RGB))
plt.title('Query Image')
plt.axis('off')
plt.show()
# Display top 3 similar images
top_k = 3
for i in range(min(top_k, len(dataset_imgs))):
idx = ranked_indices[i]
plt.figure(figsize=(5,5))
plt.imshow(cv2.cvtColor(dataset_imgs[idx], cv2.COLOR_BGR2RGB))
plt.title(f"Similar Image {i+1} (Score: {similarities[idx]:.2f})")
plt.axis('off')
plt.show()

IndentationError: expected an indented block after function definition on line 14 (709289347.py, line 15)